# DelhiCold Logistics — Late Delivery Early-Warning Model

This notebook builds, explains, and evaluates the late-delivery prediction pipeline for **DelhiCold Logistics** across **12 Delhi/NCR cold-chain hubs**.

### Workflow
1. **Data Generation & Quality Audit**: Mixed time zones (`+05:30`, `UTC`, naive IST, `+04:00`), out-of-order warehouse scan events, and legacy hub renames after the `2025-07-01` network change.
2. **Deterministic Cleaning & Chronological Split**: Leak-free 70% / 15% / 15% time-based split.
3. **4-Model Benchmark + Baseline**: Dummy baseline vs. Logistic Regression, Random Forest, XGBoost, and LightGBM with class-imbalance handling.
4. **Hyperparameter Tuning**: `RandomizedSearchCV` with `TimeSeriesSplit`.
5. **Business Cost-Matrix Threshold Optimization**: Minimizing expected INR cost (`FN = ₹5,000`, `FP = ₹400`, `TP = ₹400`, `TN = ₹0`).
6. **Global & Individual SHAP Explainability** + **Feature Drift Monitoring (PSI & KS-test)**.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import CANONICAL_HUBS, COST_MATRIX_INR, MODEL_PATH, METADATA_PATH
from src.data_generator import generate_coldchain_dataset
from src.preprocessing import clean_and_engineer_features, chronological_split, get_feature_columns
from src.threshold import optimize_threshold, evaluate_threshold_cost
from src.drift import compute_reference_stats, detect_drift
from src.train import run_training

sns.set_theme(style='whitegrid')
N_ROWS = int(os.getenv('TRAIN_ROWS', '100000'))  # Set TRAIN_ROWS=1500000 for full 1.5M-row run
print(f'Configured dataset size: {N_ROWS:,} rows')

Configured dataset size: 1,500,000 rows


## 1. Data Generation & Data Quality Diagnostics

In [2]:
raw_df, scans_df = generate_coldchain_dataset(n_rows=N_ROWS, seed=42)
print('Raw shipments shape:', raw_df.shape)
print('Sample warehouse scans shape:', scans_df.shape)
print('Unique raw origin hub codes (including pre-network-change legacy names):', raw_df['origin_hub_raw'].nunique())
print('Source timezone breakdown:\n', raw_df['source_timezone'].value_counts(normalize=True).round(3))
print('Out-of-order scan rate:', f"{raw_df['out_of_order_scan_flag'].mean():.2%}")
print('Target late-delivery rate:', f"{raw_df['is_late'].mean():.2%}")
raw_df.head(3)

Raw shipments shape: (1500000, 22)
Sample warehouse scans shape: (25000, 5)
Unique raw origin hub codes (including pre-network-change legacy names): 24
Source timezone breakdown:
 source_timezone
Asia/Kolkata    0.50
UTC             0.25
NAIVE_IST       0.20
Asia/Dubai      0.05
Name: proportion, dtype: float64
Out-of-order scan rate: 12.02%
Target late-delivery rate: 20.31%


,shipment_id,vehicle_id,origin_hub_raw,dest_hub_raw,dispatch_time_raw,planned_delivery_raw,actual_delivery_raw,source_timezone,cargo_type,vehicle_type,...,planned_transit_hours,warehouse_dwell_hours,scan_count,out_of_order_scan_flag,temp_setpoint_c,avg_cabin_temp_c,vehicle_age_years,payload_weight_kg,route_congestion_index,is_late
0,SHP-0000001,DL1C-9221,NOI-SEC63-LEGACY,GZP-EAST-HUB,2025-01-01T05:30:03+05:30,2025-01-01T06:46:15+05:30,2025-01-01T06:15:50+05:30,Asia/Kolkata,fresh_dairy,insulated_lcv,...,1.27,1.46,3,0,3.0,4.60,4.2,1541.4,0.814,0
1,SHP-0000002,DL1C-8107,HUB-HR-GGN,AZADPUR-MKT,2025-01-01T04:00:27+04:00,2025-01-01T06:19:03+04:00,2025-01-01T10:03:37+04:00,Asia/Dubai,fresh_produce,reefer_truck_7t,...,2.31,0.66,5,0,7.0,7.92,12.3,1221.5,0.841,1
2,SHP-0000003,DL1C-1358,GZP-EAST-HUB,GZB-SAHIBABAD-01,2025-01-01T05:30:42+05:30,2025-01-01T06:46:54+05:30,2025-01-01T06:31:55+05:30,Asia/Kolkata,frozen_meat,reefer_truck_3t,...,1.27,1.11,4,0,-18.0,-15.95,2.1,2060.1,0.660,0


## 2. Cleaning, Canonicalization, Feature Engineering & Time-Based Split

In [3]:
clean_df = clean_and_engineer_features(raw_df, scans_df)
assert clean_df['origin_hub_id'].nunique() == 12
assert clean_df['dispatch_time_utc'].isna().sum() == 0

train_df, val_df, test_df = chronological_split(clean_df)
print(f'Train window: {train_df["dispatch_time_utc"].min()} -> {train_df["dispatch_time_utc"].max()} ({len(train_df):,} rows)')
print(f'Val window:   {val_df["dispatch_time_utc"].min()} -> {val_df["dispatch_time_utc"].max()} ({len(val_df):,} rows)')
print(f'Test window:  {test_df["dispatch_time_utc"].min()} -> {test_df["dispatch_time_utc"].max()} ({len(test_df):,} rows)')

Train window: 2025-01-01 00:00:03+00:00 -> 2025-09-13 05:54:34+00:00 (1,050,000 rows)
Val window:   2025-09-13 05:54:59+00:00 -> 2025-11-07 01:23:53+00:00 (225,000 rows)
Test window:  2025-11-07 01:24:17+00:00 -> 2025-12-31 23:59:51+00:00 (225,000 rows)


## 3. Train 4 Models, Tune Winner with RandomizedSearchCV & Save Artifacts

In [4]:
metadata = run_training(n_rows=N_ROWS, tune_iter=10, seed=42)
leaderboard_df = pd.DataFrame(metadata['leaderboard_val']).T.sort_values('pr_auc', ascending=False)
display(leaderboard_df)
print(f"Best model: {metadata['best_model_family']} | Test PR-AUC lift over baseline: +{metadata['pr_auc_lift_vs_baseline_pct']}%")

[1/7] Generating 1,500,000 Delhi cold-chain shipment records (12 hubs)...


[2/7] Cleaning time zones, out-of-order scans, hub renames & engineering features...


[3/7] Performing strict chronological train (70%) / val (15%) / test (15%) split...


[4/7] Training 4 candidate models with class-imbalance handling...


   - logistic_regression  | Val PR-AUC=0.5840 | ROC-AUC=0.8095 | 7.98s


   - random_forest        | Val PR-AUC=0.5758 | ROC-AUC=0.8088 | 36.53s


... (3 outputs left out)

[6/7] Optimizing decision threshold using cold-chain business cost matrix (INR)...
   - Optimal Threshold=0.28 | Test PR-AUC=0.5837 (+183.8% vs baseline 0.2057)
   - Test Cost @ 0.28: INR 79,300,800 (saves INR 152,759,200 vs do-nothing, INR 20,638,800 vs 0.50 threshold)
[7/7] Computing global SHAP explanations, drift reference stats & saving artifacts...


Done in 1374.66s! Saved model -> /app/models/late_delivery_model.joblib


,pr_auc,roc_auc,f1_at_050,precision_at_050,recall_at_050,train_seconds
xgboost,0.5885,0.8133,0.5369,0.4340,0.7038,40.31
lightgbm,0.5879,0.8130,0.5370,0.4334,0.7057,200.65
logistic_regression,0.5840,0.8095,0.5290,0.4192,0.7167,7.98
random_forest,0.5758,0.8088,0.5367,0.4426,0.6816,36.53
baseline_dummy,0.2018,0.5009,0.2037,0.2029,0.2046,NaN


Best model: xgboost | Test PR-AUC lift over baseline: +183.76%


## 4. Business Cost-Matrix Threshold Optimization

In [5]:
import joblib

model = joblib.load(MODEL_PATH)
feature_cols = get_feature_columns()
val_probs = model.predict_proba(val_df[feature_cols])[:, 1]
thresh_report = optimize_threshold(val_df['is_late'].values, val_probs, COST_MATRIX_INR)

curve_df = pd.DataFrame(thresh_report['threshold_curve'])
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(curve_df['threshold'], curve_df['cost_per_shipment_inr'], color='#0B5CAD', linewidth=2)
ax.axvline(thresh_report['optimal_threshold'], color='#D9381E', linestyle='--', label=f"Optimal ({thresh_report['optimal_threshold']:.2f})")
ax.axvline(0.50, color='#666666', linestyle=':', label='Default (0.50)')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Expected Cost per Shipment (INR)')
ax.set_title('Cold-Chain Business Cost vs. Decision Threshold')
ax.legend()
plt.tight_layout()
plt.show()

print('Optimal threshold metrics (Test Set):', metadata['test_cost_at_optimal_threshold'])
print('Default 0.50 metrics (Test Set):     ', metadata['test_cost_at_default_050'])

Optimal threshold metrics (Test Set): {'threshold': 0.28, 'tp': 42466, 'fp': 106461, 'fn': 3946, 'tn': 72127, 'precision': 0.2851, 'recall': 0.915, 'f1': 0.4348, 'total_cost_inr': 79300800.0, 'cost_per_shipment_inr': 352.45, 'do_nothing_cost_inr': 232060000.0, 'savings_vs_do_nothing_inr': 152759200.0, 'savings_pct_vs_do_nothing': 65.83}
Default 0.50 metrics (Test Set):      {'threshold': 0.5, 'tp': 32506, 'fp': 43518, 'fn': 13906, 'tn': 135070, 'precision': 0.4276, 'recall': 0.7004, 'f1': 0.531, 'total_cost_inr': 99939600.0, 'cost_per_shipment_inr': 444.18, 'do_nothing_cost_inr': 232060000.0, 'savings_vs_do_nothing_inr': 132120400.0, 'savings_pct_vs_do_nothing': 56.93}


## 5. Global & Individual SHAP Explanations

In [6]:
from src.explainability import explain_single_shipment

global_shap_df = pd.DataFrame(metadata['top_global_shap_features'])
display(global_shap_df)

# Local explanation for one high-risk test shipment
test_probs = model.predict_proba(test_df[feature_cols])[:, 1]
high_risk_idx = int(np.argmax(test_probs))
single_row = test_df[feature_cols].iloc[[high_risk_idx]]
local_exp = explain_single_shipment(model, single_row, top_k=8)
print(f'Highest-risk test shipment probability: {test_probs[high_risk_idx]:.4f}')
pd.DataFrame(local_exp['top_contributions'])

,feature,mean_abs_shap
0,warehouse_dwell_hours,0.63120
1,weather_condition_clear,0.33920
2,is_peak_hour,0.26284
3,route_congestion_index,0.17615
4,dwell_to_transit_ratio,0.15554
5,vehicle_age_years,0.13499
6,weather_condition_heavy_rain_monsoon,0.12372
7,weather_condition_dense_fog,0.11578
8,weather_condition_haze_smog,0.10853
9,vehicle_type_reefer_van_1t,0.08955


Highest-risk test shipment probability: 0.9951


,feature,shap_value,direction
0,warehouse_dwell_hours,2.64947,increases_late_risk
1,dwell_to_transit_ratio,0.91576,increases_late_risk
2,weather_condition_heavy_rain_monsoon,0.41113,increases_late_risk
3,route_congestion_index,0.37151,increases_late_risk
4,is_peak_hour,0.22537,increases_late_risk
5,out_of_order_scan_flag,0.22249,increases_late_risk
6,weather_condition_clear,0.19704,increases_late_risk
7,vehicle_age_years,0.11537,increases_late_risk


## 6. Stretch Goal — Drift Monitoring Simulation (Monsoon / Heatwave Shift)

In [7]:
ref_stats = compute_reference_stats(train_df)
shifted_df = test_df.sample(n=min(len(test_df), 2000), random_state=42).copy()
shifted_df['warehouse_dwell_hours'] = shifted_df['warehouse_dwell_hours'] + 2.2
shifted_df['route_congestion_index'] = np.clip(shifted_df['route_congestion_index'] + 0.25, 0, 1)

drift_report = detect_drift(shifted_df, ref_stats, log_warnings=True)
print('Overall Drift Status:', drift_report['overall_status'])
print('Drifted Features:', drift_report['drifted_features'])
pd.DataFrame(drift_report['features']).T

DATA DRIFT WARNING | feature=warehouse_dwell_hours | PSI=10.9187 | KS_stat=0.8485 | KS_p=0 | ref_mean=1.493 | cur_mean=3.720


DATA DRIFT WARNING | feature=route_congestion_index | PSI=4.7400 | KS_stat=0.6823 | KS_p=0 | ref_mean=0.717 | cur_mean=0.932


CATEGORICAL DRIFT WARNING | feature=weather_condition | PSI=0.8402


Overall Drift Status: DRIFT_WARNING
Drifted Features: ['warehouse_dwell_hours', 'route_congestion_index', 'weather_condition']


,type,psi,ks_stat,ks_pvalue,ref_mean,current_mean,status
distance_km,numeric,0.0021,0.0285,0.391087,47.6,47.6253,STABLE
planned_transit_hours,numeric,0.0041,0.0415,0.063831,2.3776,2.3816,STABLE
warehouse_dwell_hours,numeric,10.9187,0.8485,0.0,1.4929,3.7196,DRIFT_DETECTED
temp_deviation_c,numeric,0.0123,0.0557,0.004344,0.992,0.9306,STABLE
route_congestion_index,numeric,4.74,0.6823,0.0,0.7167,0.9318,DRIFT_DETECTED
vehicle_age_years,numeric,0.0076,0.0232,0.650142,4.3062,4.2816,STABLE
payload_weight_kg,numeric,0.0041,0.032,0.254008,1814.4934,1787.6837,STABLE
dwell_to_transit_ratio,numeric,0.0026,0.023,0.663087,0.6961,0.7085,STABLE
origin_hub_id,categorical,0.0055,NaN,NaN,NaN,NaN,STABLE
dest_hub_id,categorical,0.0054,NaN,NaN,NaN,NaN,STABLE
